# Main79 SVC-Bag — FINAL VERIFIED 3x3 ALL-IN-ONE COLAB RUN

**Purpose:** test the architectural change we actually discussed: replace the single LinearSVC inside the Main66 classical teacher with a small 3-model bootstrap bag, propagate that teacher through the exact Main79 residual architecture, and then compare against the locked Main79 / Main79+Exp11 references.

This version is deliberately reduced for the deadline: **3 SVC bag members × 3 strict OOF folds = 9 SVC fits**, instead of the previous 10 × 5 = 50 fits. The OOF SVC signal remains leakage-safe; every training row receives its bagged SVC score from a fold whose SVCs did not train on that row.

The friend-provided `META_C=3.0` information is used for the teacher LogisticRegression. The teacher-only `0.595` threshold is reported diagnostically but is **not** used for the final Main79 decision; Main79 still thresholds its final logit at 0.

## COLAB INPUT
Upload **one ZIP** through the Colab **Files** sidebar before Run All. The ZIP must contain the following files at its top level (nested paths are also detected):

- `train.json`
- `test.json`
- `main66.py`
- `main79_95_kaggle.py`
- `main64_oof_meta_features.npy`
- `main64_val_meta_features.npy`
- `main64_oof_svm.npy`
- `main64_oof_nbsvm.npy`
- `main64_oof_hgb.npy`
- `main64_oof_local.npy`

The notebook does **not** require Kaggle and contains no Kaggle-specific paths.

## OUTPUT
Everything is saved under `/content/main79_svcbag_fast_outputs/`; the final package is `/content/main79_svcbag_fast_outputs.zip`.


**Important:** This notebook preserves the successful FAST 3x3 experiment. It does not use the abandoned 10×5/50-fit SVC configuration. The Section-8 VOCAB_SIZE/PAD_IDX direct-call hotfix is integrated below.


In [1]:
# ============================================================
# 0. COLAB INPUT CHECK — REQUIRED FILES
# ============================================================
from pathlib import Path
import os, json, zipfile, shutil, random, time, importlib.util, sys, warnings, gc

import numpy as np
import pandas as pd

CONTENT = Path('/content')
OUTPUT_DIR = CONTENT / 'main79_svcbag_fast_outputs'
RUNTIME = CONTENT / 'main79_svcbag_fast_runtime'

REQUIRED_NAMES = {
    'train.json',
    'test.json',
    'main66.py',
    'main79_95_kaggle.py',
    'main64_oof_meta_features.npy',
    'main64_val_meta_features.npy',
    'main64_oof_svm.npy',
    'main64_oof_nbsvm.npy',
    'main64_oof_hgb.npy',
    'main64_oof_local.npy',
}

for d in [OUTPUT_DIR, RUNTIME]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)


def archive_names(zp):
    with zipfile.ZipFile(zp, 'r') as z:
        return {Path(n).name for n in z.namelist() if not n.endswith('/')}

# Prefer a ZIP containing all required files.
candidates = sorted(CONTENT.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
selected_zip = None
for zp in candidates:
    try:
        if REQUIRED_NAMES.issubset(archive_names(zp)):
            selected_zip = zp
            break
    except Exception:
        pass

if selected_zip is not None:
    print('SOURCE ZIP:', selected_zip)
    with zipfile.ZipFile(selected_zip, 'r') as z:
        z.extractall(RUNTIME)
else:
    print('No suitable ZIP found; searching /content for individual required files.')

all_files = {}
for p in CONTENT.rglob('*'):
    if p.is_file() and p.name in REQUIRED_NAMES:
        all_files.setdefault(p.name, p)

missing = sorted(REQUIRED_NAMES - set(all_files))
if missing:
    raise FileNotFoundError(
        'Missing required files. Upload one ZIP containing:\n' +
        '\n'.join('  - ' + x for x in sorted(REQUIRED_NAMES))
    )

FILES_DIR = RUNTIME / 'files'
FILES_DIR.mkdir(parents=True, exist_ok=True)
SOURCE = {}
for name in sorted(REQUIRED_NAMES):
    dst = FILES_DIR / name
    shutil.copy2(all_files[name], dst)
    SOURCE[name] = dst
    print('OK:', name)

for name in ['main66.py', 'main79_95_kaggle.py']:
    shutil.copy2(SOURCE[name], OUTPUT_DIR / name)

print('\nOUTPUT_DIR =', OUTPUT_DIR)


SOURCE ZIP: /content/Main79_SVC_Bag_FAST_3x3_Colab_bundle.zip
OK: main64_oof_hgb.npy
OK: main64_oof_local.npy
OK: main64_oof_meta_features.npy
OK: main64_oof_nbsvm.npy
OK: main64_oof_svm.npy
OK: main64_val_meta_features.npy
OK: main66.py
OK: main79_95_kaggle.py
OK: test.json
OK: train.json

OUTPUT_DIR = /content/main79_svcbag_fast_outputs


## 1. Imports, source loading, canonical split, and frozen references

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')

SEED = 42

# FAST CONTROLLED EXPERIMENT:
# 3 SVC members × 3 strict OOF folds = 9 SVC fits.
SVC_BAG_SEEDS = [42, 123, 777]
SVC_BAG_SIZE = len(SVC_BAG_SEEDS)
SVC_BOOTSTRAP_FRACTION = 1.0
SVC_OOF_FOLDS = 3

# Original Main66 SVC strength remains unchanged.
SVC_C = 10.0

# Friend-provided teacher-meta experiment: use META_C=3.0.
META_C_EXPERIMENT = 3.0
TEACHER_DIAGNOSTIC_THRESHOLD = 0.595

LOCKED_MAIN79_ACC = 0.9364326375711575
LOCKED_MAIN79_AUC = 0.9803698435277381
LOCKED_BLEND_ACC = 0.9388045540796964
LOCKED_BLEND_AUC = 0.9826665481270744
LOCKED_PUBLIC_MAIN79 = 0.9500
LOCKED_PUBLIC_BLEND = 0.9506


def import_module_from_file(module_name, path):
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f'Could not load module from {path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def load_jsonl(path, labelled=True):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    ids = [r['id'] for r in rows]
    texts = [r['text'] for r in rows]
    if labelled:
        y = np.asarray([0 if r['label'] == 'A' else 1 for r in rows], dtype=np.int64)
        return ids, texts, y
    return ids, texts

train_ids, texts, labels = load_jsonl(SOURCE['train.json'], True)
test_ids, test_texts = load_jsonl(SOURCE['test.json'], False)

idx = np.arange(len(labels))
train_idx, val_idx = train_test_split(
    idx, test_size=0.20, random_state=SEED, stratify=labels
)
train_idx = np.asarray(train_idx, dtype=np.int64)
val_idx = np.asarray(val_idx, dtype=np.int64)
y_train = labels[train_idx]
y_val = labels[val_idx]
train_texts = [texts[i] for i in train_idx]
val_texts = [texts[i] for i in val_idx]

main66 = import_module_from_file('main66_uploaded_fast', SOURCE['main66.py'])
main79 = import_module_from_file('main79_uploaded_fast', SOURCE['main79_95_kaggle.py'])

# Use the friend's teacher-meta C only for this experiment.
main79.META_C = META_C_EXPERIMENT

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Train:', len(train_texts), '| Validation:', len(val_texts), '| Full train:', len(texts), '| Test:', len(test_texts))
print('SVC bag:', SVC_BAG_SIZE, 'members | OOF folds:', SVC_OOF_FOLDS, '| META_C:', META_C_EXPERIMENT)

assert len(texts) == 10536
assert len(train_idx) == 8428
assert len(val_idx) == 2108
assert len(test_texts) == 3000
assert main66.SVM_C == 10.0
assert main79.MODEL_SEEDS == [42, 123, 777]
assert main79.EPOCHS == 6
assert main79.RESIDUAL_WEIGHT == 0.025

print('\nLOCKED REFERENCES:')
print(f'Main79       = {LOCKED_MAIN79_ACC:.6f} | AUC {LOCKED_MAIN79_AUC:.6f} | public {LOCKED_PUBLIC_MAIN79:.4f}')
print(f'Main79+Exp11 = {LOCKED_BLEND_ACC:.6f} | AUC {LOCKED_BLEND_AUC:.6f} | public {LOCKED_PUBLIC_BLEND:.4f}')


PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
Train: 8428 | Validation: 2108 | Full train: 10536 | Test: 3000
SVC bag: 3 members | OOF folds: 3 | META_C: 3.0

LOCKED REFERENCES:
Main79       = 0.936433 | AUC 0.980370 | public 0.9500
Main79+Exp11 = 0.938805 | AUC 0.982667 | public 0.9506


## 2. Verify the existing Main64 OOF artifacts before changing only the SVC component

In [ ]:

# Existing OOF components from the supplied, already-validated Main64 stack.
oof_meta_orig = np.load(SOURCE['main64_oof_meta_features.npy']).astype(np.float32)
val_meta_orig = np.load(SOURCE['main64_val_meta_features.npy']).astype(np.float32)
oof_svm_orig = np.load(SOURCE['main64_oof_svm.npy']).reshape(-1)
oof_nbsvm = np.load(SOURCE['main64_oof_nbsvm.npy']).reshape(-1)
oof_hgb = np.load(SOURCE['main64_oof_hgb.npy']).reshape(-1)
oof_local = np.load(SOURCE['main64_oof_local.npy'])
if oof_local.ndim == 1:
    oof_local = oof_local[:, None]

assert oof_meta_orig.shape == (8428, 13)
assert val_meta_orig.shape == (2108, 13)
assert oof_svm_orig.shape == (8428,)
assert oof_nbsvm.shape == (8428,)
assert oof_hgb.shape == (8428,)
assert oof_local.shape[0] == 8428

# Reconstruct the original 12 standardized base columns exactly.
oof_local_b = -oof_local
oof_raw_orig = np.column_stack([
    oof_svm_orig,
    oof_nbsvm,
    oof_hgb,
    oof_local_b[:, 0],
    oof_local_b[:, 1],
    oof_local_b[:, 2],
    oof_local_b[:, 3],
    oof_local_b[:, 6],
    oof_local_b[:, 7],
    oof_local_b[:, 8],
    oof_local_b[:, 9],
    oof_local_b[:, 10],
])

orig_scaler = StandardScaler().fit(oof_raw_orig)
orig_reconstructed = orig_scaler.transform(oof_raw_orig)
orig_max_diff = float(np.max(np.abs(orig_reconstructed - oof_meta_orig[:, :12])))

print('Original Main64 meta reconstruction max diff:', orig_max_diff)
assert orig_max_diff < 1e-5, 'Existing Main64 OOF artifacts failed exact reconstruction.'
print('PASS: the supplied OOF arrays are aligned to the canonical 8428-row training split.')


Original Main64 meta reconstruction max diff: 9.301141723483397e-07
PASS: the supplied OOF arrays are aligned to the canonical 8428-row training split.


## 3. Bagged LinearSVC replacement — FAST strict OOF
We use **3 SVC members × 3 OOF folds = 9 fits**. The other Main66 components use the already supplied strict OOF arrays. This is the deliberate deadline-safe reduction from the failed 10×5 run.

Each OOF fold fits its TF-IDF/transition representation only on that fold's training rows. The held-out rows are never used to fit those SVCs.


In [ ]:
def make_bootstrap_indices(n, seed, fraction=1.0):
    rng = np.random.RandomState(seed)
    m = max(1, int(round(fraction * n)))
    return rng.randint(0, n, size=m)


def fit_svc_bag(X_fit, y_fit, X_query, seeds=SVC_BAG_SEEDS, tag='bag'):
    scores = []
    n = len(y_fit)
    for k, seed in enumerate(seeds, 1):
        boot = make_bootstrap_indices(n, seed, SVC_BOOTSTRAP_FRACTION)
        model = LinearSVC(
            C=SVC_C,
            class_weight='balanced',
            tol=1e-2,
            max_iter=500000,
            random_state=seed,
        )
        t0 = time.time()
        model.fit(X_fit[boot], y_fit[boot])
        score = model.decision_function(X_query).astype(np.float32)
        scores.append(score)
        print(f'[{tag}] SVC {k:02d}/{len(seeds)} seed={seed} | time={(time.time()-t0):.2f}s')
        del model
        gc.collect()
    return np.mean(np.vstack(scores), axis=0).astype(np.float32)


# Strict OOF: 3 folds, so every training row gets a score from SVCs that did not train on it.
svc_bag_oof = np.empty(len(train_idx), dtype=np.float32)
outer = StratifiedKFold(n_splits=SVC_OOF_FOLDS, shuffle=True, random_state=SEED)

t0_all = time.time()
for fold, (fit_pos, hold_pos) in enumerate(outer.split(train_texts, y_train), 1):
    print('\n' + '=' * 72)
    print(f'SVC BAG OOF FOLD {fold}/{SVC_OOF_FOLDS}')
    print('=' * 72)

    fold_fit_texts = [train_texts[i] for i in fit_pos]
    fold_hold_texts = [train_texts[i] for i in hold_pos]
    fold_y = y_train[fit_pos]

    (
        _, _, fold_tfidf, fold_trans_vectorizer, fold_compact_scaler,
        _, _, fold_X_global,
    ) = main66.fit_global_representation(fold_fit_texts)

    (
        _, _, _, _, fold_X_hold_global,
    ) = main66.transform_global_representation(
        fold_hold_texts, fold_tfidf, fold_trans_vectorizer, fold_compact_scaler
    )

    # Offset seeds by fold so every fitted SVC is genuinely distinct.
    fold_seeds = [s + (fold - 1) * 10000 for s in SVC_BAG_SEEDS]
    fold_score = fit_svc_bag(
        fold_X_global, fold_y, fold_X_hold_global,
        seeds=fold_seeds, tag=f'OOF fold {fold}'
    )
    svc_bag_oof[hold_pos] = fold_score
    print('Fold score range:', float(fold_score.min()), float(fold_score.max()))

    del fold_tfidf, fold_trans_vectorizer, fold_compact_scaler, fold_X_global, fold_X_hold_global
    gc.collect()

print('\n' + '=' * 72)
print('STRICT OOF SVC BAG COMPLETE')
print('=' * 72)
print('OOF shape:', svc_bag_oof.shape)
print('Total OOF SVC time (min):', (time.time()-t0_all)/60.0)
np.save(OUTPUT_DIR / 'svc_bag_oof.npy', svc_bag_oof)



SVC BAG OOF FOLD 1/3
[OOF fold 1] SVC 01/3 seed=42 | time=124.41s


## 4. Build the new bagged Main66 teacher for validation

In [ ]:

# Recompute the validation-side Main66 components exactly from the supplied source,
# because the provided validation meta file contains the OLD single-SVC component.
print('=' * 72)
print('BUILD VALIDATION-SIDE MAIN66 COMPONENTS')
print('=' * 72)

(
    tr_strings,
    tr_transitions,
    tr_tfidf,
    tr_trans_vectorizer,
    tr_compact_scaler,
    X_tr_tfidf,
    X_tr_compact,
    X_tr_global,
) = main66.fit_global_representation(train_texts)

(
    va_strings,
    va_transitions,
    X_va_tfidf,
    X_va_compact,
    X_va_global,
) = main66.transform_global_representation(
    val_texts,
    tr_tfidf,
    tr_trans_vectorizer,
    tr_compact_scaler,
)

print('Validation representation:', X_tr_global.shape, '->', X_va_global.shape)

# New bagged SVC validation signal.
svc_bag_val = fit_svc_bag(
    X_tr_global,
    y_train,
    X_va_global,
    seeds=SVC_BAG_SEEDS,
    tag='VAL',
)

# Existing Main66 non-SVC validation components, recomputed from the exact source.
val_nbsvm = main66.train_nbsvm(tr_strings, y_train, va_strings)
val_hgb = main66.train_hgb(X_tr_compact, y_train, X_va_compact)
val_local = main66.local_geometry_features(
    X_tr_tfidf,
    y_train,
    X_va_tfidf,
    k=main66.LOCAL_K,
)

print('Validation NB-SVM:', val_nbsvm.shape)
print('Validation HGB:', val_hgb.shape)
print('Validation local:', val_local.shape)


def make_teacher_meta(svc, nbsvm, hgb, local, scaler):
    local_b = -np.asarray(local)
    raw = np.column_stack([
        np.asarray(svc).reshape(-1),
        np.asarray(nbsvm).reshape(-1),
        np.asarray(hgb).reshape(-1),
        local_b[:, 0],
        local_b[:, 1],
        local_b[:, 2],
        local_b[:, 3],
        local_b[:, 6],
        local_b[:, 7],
        local_b[:, 8],
        local_b[:, 9],
        local_b[:, 10],
    ])
    scaled = scaler.transform(raw)
    uncertainty = np.exp(-np.abs(np.asarray(svc).reshape(-1)))
    interaction = (local_b[:, 3] * uncertainty).reshape(-1, 1)
    return np.hstack([scaled, interaction]).astype(np.float32), raw


# New training-side meta matrix: bagged SVC OOF + existing strict OOF NB-SVM/HGB/local.
bag_oof_meta_scaler = StandardScaler().fit(
    np.column_stack([
        svc_bag_oof,
        oof_nbsvm,
        oof_hgb,
        oof_local_b[:, 0],
        oof_local_b[:, 1],
        oof_local_b[:, 2],
        oof_local_b[:, 3],
        oof_local_b[:, 6],
        oof_local_b[:, 7],
        oof_local_b[:, 8],
        oof_local_b[:, 9],
        oof_local_b[:, 10],
    ])
)

bag_oof_meta, bag_oof_raw = make_teacher_meta(
    svc_bag_oof,
    oof_nbsvm,
    oof_hgb,
    oof_local,
    bag_oof_meta_scaler,
)
bag_val_meta, bag_val_raw = make_teacher_meta(
    svc_bag_val,
    val_nbsvm,
    val_hgb,
    val_local,
    bag_oof_meta_scaler,
)

assert bag_oof_meta.shape == (8428, 13)
assert bag_val_meta.shape == (2108, 13)

# Train the same Main79 teacher meta-model configuration.
teacher_model_bag = LogisticRegression(
    C=META_C_EXPERIMENT,
    max_iter=5000,
    solver='lbfgs',
    random_state=main79.SEED,
)
teacher_model_bag.fit(bag_oof_meta, y_train)

teacher_train_raw_bag = teacher_model_bag.decision_function(bag_oof_meta).astype(np.float32)
teacher_val_raw_bag = teacher_model_bag.decision_function(bag_val_meta).astype(np.float32)

teacher_scale_bag = np.percentile(np.abs(teacher_train_raw_bag), 95)
if teacher_scale_bag < 1e-6:
    teacher_scale_bag = 1.0

teacher_train_bag = np.clip(teacher_train_raw_bag / teacher_scale_bag, -6.0, 6.0).astype(np.float32)
teacher_val_bag = np.clip(teacher_val_raw_bag / teacher_scale_bag, -6.0, 6.0).astype(np.float32)

meta_mean_bag = bag_oof_meta.mean(axis=0)
meta_std_bag = np.where(bag_oof_meta.std(axis=0) < 1e-6, 1.0, bag_oof_meta.std(axis=0))
meta_train_z_bag = np.clip((bag_oof_meta - meta_mean_bag) / meta_std_bag, -6.0, 6.0).astype(np.float32)
meta_val_z_bag = np.clip((bag_val_meta - meta_mean_bag) / meta_std_bag, -6.0, 6.0).astype(np.float32)

teacher_val_acc_bag = accuracy_score(y_val, teacher_val_raw_bag >= 0)
teacher_val_auc_bag = roc_auc_score(y_val, 1.0 / (1.0 + np.exp(-teacher_val_raw_bag)))

print('\nNEW BAGGED TEACHER')
print('Teacher validation accuracy:', teacher_val_acc_bag)
print('Teacher validation AUC:', teacher_val_auc_bag)
print('Teacher scale:', float(teacher_scale_bag))
print('Bagged SVC validation mean/std:', float(svc_bag_val.mean()), float(svc_bag_val.std()))

np.save(OUTPUT_DIR / 'svc_bag_val.npy', svc_bag_val)
np.save(OUTPUT_DIR / 'bagged_teacher_val_meta.npy', bag_val_meta)


BUILD VALIDATION-SIDE MAIN66 COMPONENTS
Validation representation: (8428, 476325) -> (2108, 476325)
[VAL] SVC 01/3 seed=42 | time=614.57s
[VAL] SVC 02/3 seed=123 | time=287.63s
[VAL] SVC 03/3 seed=777 | time=98.87s
Validation NB-SVM: (2108,)
Validation HGB: (2108,)
Validation local: (2108, 11)

NEW BAGGED TEACHER
Teacher validation accuracy: 0.9293168880455408
Teacher validation AUC: 0.9754820610083769
Teacher scale: 10.619359970092773
Bagged SVC validation mean/std: 0.6836344003677368 1.9867651462554932


## 5. Train the exact Main79 residual ensemble on the canonical 8428/2108 split

In [5]:

# Exact Main79 architecture/configuration from the supplied script.
# We use its ResidualDataset, ResidualBiGRU, train_one_seed and predict_residual functions.

VOCAB_SIZE = max(int(t) for seq in texts for t in seq) + 1
PAD_IDX = VOCAB_SIZE
main79.VOCAB_SIZE = VOCAB_SIZE
main79.PAD_IDX = PAD_IDX
main79.DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
main79.USE_AMP = (main79.DEVICE.type == 'cuda')
main79.OUT_DIR = OUTPUT_DIR / 'main79_validation_models'
main79.OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Main79 DEVICE:', main79.DEVICE)
print('VOCAB_SIZE:', VOCAB_SIZE, 'PAD_IDX:', PAD_IDX)
print('Residual seeds:', main79.MODEL_SEEDS)
print('Epochs per seed:', main79.EPOCHS)
print('Residual weight:', main79.RESIDUAL_WEIGHT)

train_ds = main79.ResidualDataset(train_texts, y_train, meta_train_z_bag)
train_loader = DataLoader(
    train_ds,
    batch_size=main79.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(main79.DEVICE.type == 'cuda'),
)

val_residuals = []
val_models = []
for seed in main79.MODEL_SEEDS:
    model = main79.train_one_seed(seed, train_loader, meta_train_z_bag.shape[1])
    r = main79.predict_residual(model, val_texts, meta_val_z_bag, teacher_val_bag)
    val_models.append(model)
    val_residuals.append(r)

val_residual_mean_bag = np.mean(np.vstack(val_residuals), axis=0).astype(np.float32)
main79_svcbag_val_score = (teacher_val_bag + main79.RESIDUAL_WEIGHT * val_residual_mean_bag).astype(np.float32)
main79_svcbag_val_prob = 1.0 / (1.0 + np.exp(-main79_svcbag_val_score))
main79_svcbag_val_pred = (main79_svcbag_val_score >= 0).astype(np.int64)

main79_svcbag_val_acc = accuracy_score(y_val, main79_svcbag_val_pred)
main79_svcbag_val_auc = roc_auc_score(y_val, main79_svcbag_val_prob)

print('\n' + '=' * 72)
print('NEW MAIN79 WITH BAGGED SVC — VALIDATION')
print('=' * 72)
print('Accuracy:', main79_svcbag_val_acc)
print('AUC:', main79_svcbag_val_auc)
print('Delta vs locked Main79 accuracy:', main79_svcbag_val_acc - LOCKED_MAIN79_ACC)
print('Delta vs locked Main79 AUC:', main79_svcbag_val_auc - LOCKED_MAIN79_AUC)
print('Errors:', int(np.sum(main79_svcbag_val_pred != y_val)))

np.save(OUTPUT_DIR / 'main79_svcbag_val_score.npy', main79_svcbag_val_score)
np.save(OUTPUT_DIR / 'main79_svcbag_val_prob.npy', main79_svcbag_val_prob)
np.save(OUTPUT_DIR / 'main79_svcbag_val_predictions.npy', main79_svcbag_val_pred)
np.save(OUTPUT_DIR / 'main79_svcbag_val_labels.npy', y_val)

# Release validation models before the full-data phase.
del val_models, val_residuals
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Main79 DEVICE: cuda
VOCAB_SIZE: 18438 PAD_IDX: 18438
Residual seeds: [42, 123, 777]
Epochs per seed: 6
Residual weight: 0.025


NameError: name 'meta_train_z_bag' is not defined

## 6. Existing Exp11 branch — reproduced after the new Main79 validation

In [ ]:
# ============================================================
# EXP11 — EXACT HELPERS / ARCHITECTURE (copied from verified direct-stack source)
# ============================================================
# These definitions are copied from the uploaded Exp11 notebook,
# with no experimental parameter changes.

EXP11_SEED = 42
EXP11_MAX_LEN = 384
EXP11_EMB_DIM = 128
EXP11_CNN_CHANNELS = 128
EXP11_Z_DIM = 128
EXP11_BATCH_SIZE = 64
EXP11_EPOCHS = 15
EXP11_PATIENCE = 3
EXP11_LR = 2e-3
EXP11_WEIGHT_DECAY = 1e-4
EXP11_TFIDF_MAX_FEATURES = 200_000
EXP11_TFIDF_MIN_DF = 3
EXP11_SVD_DIM = 256

def entropy_from_counts(counts):
    c = np.asarray(counts, dtype=float)
    c = c[c > 0]
    if len(c) == 0:
        return 0.0
    p = c / c.sum()
    return float(-(p * np.log2(p)).sum())

def sequence_features(doc):
    x = list(doc)
    n = len(x)
    if n == 0:
        return np.zeros(58, dtype=np.float32)

    cnt = __import__("collections").Counter(x)
    bigrams = list(zip(x[:-1], x[1:]))
    trigrams = list(zip(x[:-2], x[1:-1], x[2:]))

    def div(seq):
        return len(set(seq)) / max(1, len(seq))

    def rep(seq):
        return 1.0 - div(seq)

    half = max(1, n // 2)
    q = max(1, n // 4)

    first = x[:half]
    second = x[half:]
    q1, q2, q3, q4 = x[:q], x[q:2*q], x[2*q:3*q], x[3*q:]

    feats = [
        n, len(cnt), len(cnt)/n,
        len([v for v in cnt.values() if v > 1])/n,
        max(cnt.values())/n,
        np.mean(list(cnt.values())),
        np.std(list(cnt.values())),
        entropy_from_counts(list(cnt.values())),
        len(bigrams), len(set(bigrams)), div(bigrams), rep(bigrams),
        len(trigrams), len(set(trigrams)), div(trigrams), rep(trigrams),
        sum(x[i] == x[i-1] for i in range(1,n))/max(1,n-1),
        len(set(zip(x[:-1], x[1:])))/max(1,n-1),
        entropy_from_counts(list(__import__("collections").Counter(bigrams).values())),
        entropy_from_counts(list(__import__("collections").Counter(trigrams).values())),
        len(set(first))/len(first),
        len(set(second))/max(1,len(second)),
        entropy_from_counts(list(__import__("collections").Counter(first).values())),
        entropy_from_counts(list(__import__("collections").Counter(second).values())),
        len(set(q1))/len(q1), len(set(q2))/len(q2),
        len(set(q3))/len(q3), len(set(q4))/max(1,len(q4)),
        entropy_from_counts(list(__import__("collections").Counter(q1).values())),
        entropy_from_counts(list(__import__("collections").Counter(q2).values())),
        entropy_from_counts(list(__import__("collections").Counter(q3).values())),
        entropy_from_counts(list(__import__("collections").Counter(q4).values())),
        len(first), len(second), len(q1), len(q2), len(q3), len(q4),
        len(set(first)&set(second))/max(1,len(set(first)|set(second))),
        len(set(q1)&set(q2))/max(1,len(set(q1)|set(q2))),
        len(set(q2)&set(q3))/max(1,len(set(q2)|set(q3))),
        len(set(q3)&set(q4))/max(1,len(set(q3)|set(q4))),
        np.mean([len(t) for t in x]),
        np.std([len(t) for t in x]),
        min([len(t) for t in x]),
        max([len(t) for t in x]),
        np.mean([len(t) for t in x[:half]]),
        np.mean([len(t) for t in x[half:]]) if second else 0.0,
        len(x[0]), len(x[-1]),
        len(set(x[:min(10,n)]))/min(10,n),
        len(set(x[max(0,n-10):]))/min(10,n),
        entropy_from_counts(list(__import__("collections").Counter(x[:min(10,n)]).values())),
        entropy_from_counts(list(__import__("collections").Counter(x[max(0,n-10):]).values())),
        len(set(x[:min(25,n)]))/min(25,n),
        len(set(x[max(0,n-25):]))/min(25,n),
        entropy_from_counts(list(__import__("collections").Counter(x[:min(25,n)]).values())),
        entropy_from_counts(list(__import__("collections").Counter(x[max(0,n-25):]).values())),
    ]

    return np.asarray(feats, dtype=np.float32)

def build_vocab(all_docs):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for doc in all_docs:
        for t in doc:
            if t not in vocab:
                vocab[t] = len(vocab)
    return vocab

def encode_docs(docs, vocab):
    out = []
    for doc in docs:
        ids = [vocab.get(t, 1) for t in doc[:EXP11_MAX_LEN]]
        ids += [0] * (EXP11_MAX_LEN - len(ids))
        out.append(ids)
    return np.asarray(out, dtype=np.int64)

class TokenDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = sequences
        self.y = np.asarray(labels, dtype=np.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return (
            torch.tensor(self.x[i], dtype=torch.long),
            torch.tensor(self.y[i])
        )

class TokenEncoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(
            vocab_size, EXP11_EMB_DIM, padding_idx=0
        )
        self.convs = nn.ModuleList([
            nn.Conv1d(
                EXP11_EMB_DIM,
                EXP11_CNN_CHANNELS,
                k,
                padding=k//2
            )
            for k in (3,5,7)
        ])
        self.proj = nn.Sequential(
            nn.Linear(EXP11_CNN_CHANNELS * 3, EXP11_Z_DIM),
            nn.GELU(),
            nn.LayerNorm(EXP11_Z_DIM),
        )
    def forward(self, x):
        h = self.emb(x).transpose(1,2)
        pooled = []
        for conv in self.convs:
            z = torch.nn.functional.gelu(conv(h))
            z = torch.max(z, dim=-1).values
            pooled.append(z)
        return self.proj(torch.cat(pooled, dim=1))

class MLPEncoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d,256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(256,EXP11_Z_DIM),
            nn.GELU(),
            nn.LayerNorm(EXP11_Z_DIM),
        )
    def forward(self,x):
        return self.net(x)

class FusionModel(nn.Module):
    def __init__(self, vocab_size, dims):
        super().__init__()
        self.token = TokenEncoder(vocab_size)
        self.branches = nn.ModuleDict()
        for name,d in dims.items():
            self.branches[name] = MLPEncoder(d)
        n = 128 * (1 + len(dims))
        self.fusion = nn.Sequential(
            nn.Linear(n,256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(256,EXP11_Z_DIM),
            nn.GELU(),
            nn.LayerNorm(EXP11_Z_DIM),
        )
        self.head = nn.Linear(EXP11_Z_DIM,1)

    def forward(self,tok,branch_inputs):
        zs = [self.token(tok)]
        for name in self.branches:
            zs.append(self.branches[name](branch_inputs[name]))
        z = self.fusion(torch.cat(zs,dim=1))
        return self.head(z).squeeze(1), z

def make_exp11_structural(docs):
    return np.vstack([sequence_features(d) for d in docs]).astype(np.float32)

def make_exp11_loaders(tok_train, ytr, branches_train,
                       tok_val, yv, branches_val):
    class MultiInputDataset(Dataset):
        def __init__(self,tok,y,b):
            self.tok=tok
            self.y=np.asarray(y,dtype=np.float32)
            self.b=b
        def __len__(self): return len(self.y)
        def __getitem__(self,i):
            return (
                torch.tensor(self.tok[i],dtype=torch.long),
                torch.tensor(self.y[i],dtype=torch.float32),
                {k:torch.tensor(v[i],dtype=torch.float32)
                 for k,v in self.b.items()}
            )

    tr = MultiInputDataset(tok_train,ytr,branches_train)
    va = MultiInputDataset(tok_val,yv,branches_val)

    return (
        DataLoader(tr,batch_size=EXP11_BATCH_SIZE,shuffle=True),
        DataLoader(va,batch_size=EXP11_BATCH_SIZE,shuffle=False)
    )


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# ============================================================
# EXP11 VALIDATION — same direct-stack implementation
# ============================================================
def run_exp11_validation():
    # Original Exp11 convention: A=1, B=0.
    exp_y_train = (labels[train_idx] == 0).astype(np.float32)
    exp_y_val = (labels[val_idx] == 0).astype(np.float32)

    docs_tr = [[str(t) for t in texts[i]] for i in train_idx]
    docs_va = [[str(t) for t in texts[i]] for i in val_idx]

    vocab = build_vocab(docs_tr)
    tok_tr = encode_docs(docs_tr, vocab)
    tok_va = encode_docs(docs_va, vocab)

    strings_tr = [' '.join(d) for d in docs_tr]
    strings_va = [' '.join(d) for d in docs_va]

    vec = TfidfVectorizer(
        ngram_range=(1,6),
        min_df=EXP11_TFIDF_MIN_DF,
        max_features=EXP11_TFIDF_MAX_FEATURES,
        sublinear_tf=True,
        token_pattern=r'(?u)\S+',
        dtype=np.float32,
    )

    A = vec.fit_transform(strings_tr)
    B = vec.transform(strings_va)

    ncomp = min(EXP11_SVD_DIM, A.shape[1] - 1)
    svd = TruncatedSVD(n_components=ncomp, random_state=EXP11_SEED)
    A_svd = svd.fit_transform(A).astype(np.float32)
    B_svd = svd.transform(B).astype(np.float32)

    Xraw = np.vstack([
        make_exp11_structural(docs_tr),
        make_exp11_structural(docs_va),
    ])
    scaler = StandardScaler()
    Xscaled = scaler.fit_transform(Xraw).astype(np.float32)

    branches_tr = {'tfidf': A_svd, 'structural': Xscaled[:len(docs_tr)]}
    branches_va = {'tfidf': B_svd, 'structural': Xscaled[len(docs_tr):]}

    tr_loader, va_loader = make_exp11_loaders(
        tok_tr, exp_y_train, branches_tr,
        tok_va, exp_y_val, branches_va,
    )

    dims = {k: v.shape[1] for k, v in branches_tr.items()}
    model = FusionModel(len(vocab), dims).to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    device = next(model.parameters()).device

    pos_weight = float((exp_y_train == 0).sum() / max(1, (exp_y_train == 1).sum()))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=EXP11_LR, weight_decay=EXP11_WEIGHT_DECAY)

    best_auc = -np.inf
    best_state = None
    best_epoch = 0
    bad = 0

    for epoch in range(1, EXP11_EPOCHS + 1):
        model.train()
        for tok, y, b in tr_loader:
            tok = tok.to(device)
            y = y.to(device)
            b = {k: v.to(device) for k, v in b.items()}
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(tok, b)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        probs, ys = [], []
        with torch.no_grad():
            for tok, y, b in va_loader:
                tok = tok.to(device)
                b = {k: v.to(device) for k, v in b.items()}
                logits, _ = model(tok, b)
                probs.extend(torch.sigmoid(logits).cpu().numpy())
                ys.extend(y.numpy())

        auc = roc_auc_score(ys, probs)
        print(f'Exp11 epoch {epoch:02d} | val AUC {auc:.5f}')

        if auc > best_auc:
            best_auc = auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= EXP11_PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    p = []
    with torch.no_grad():
        for tok, y, b in va_loader:
            tok = tok.to(device)
            b = {k: v.to(device) for k, v in b.items()}
            logits, _ = model(tok, b)
            p.extend(torch.sigmoid(logits).cpu().numpy())

    pA = np.asarray(p, dtype=np.float32)
    pB = 1.0 - pA
    acc = accuracy_score(y_val, pB >= 0.5)
    auc = roc_auc_score(y_val, pB)
    return {
        'model': model,
        'best_epoch': best_epoch,
        'val_prob_A': pA,
        'val_prob_B': pB,
        'val_acc': float(acc),
        'val_auc': float(auc),
        'vocab': vocab,
    }

exp11_val = run_exp11_validation()
exp11_val_prob_B = exp11_val['val_prob_B']

print('\n' + '=' * 72)
print('EXP11 VALIDATION RESULT')
print('=' * 72)
print('Accuracy:', exp11_val['val_acc'])
print('AUC:', exp11_val['val_auc'])
print('Best epoch:', exp11_val['best_epoch'])

np.save(OUTPUT_DIR / 'exp11_val_prob_B.npy', exp11_val_prob_B)

# We only need Exp11 validation probabilities and best_epoch from this point onward.
exp11_best_epoch = int(exp11_val['best_epoch'])
exp11_val['model'] = None
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Exp11 epoch 01 | val AUC 0.95201
Exp11 epoch 02 | val AUC 0.96131
Exp11 epoch 03 | val AUC 0.96591
Exp11 epoch 04 | val AUC 0.95862
Exp11 epoch 05 | val AUC 0.96602
Exp11 epoch 06 | val AUC 0.96348
Exp11 epoch 07 | val AUC 0.95807
Exp11 epoch 08 | val AUC 0.95651

EXP11 VALIDATION RESULT
Accuracy: 0.9060721062618596
AUC: 0.9660166745693062
Best epoch: 5


## 7. Compare the new Main79 and run the same transparent Main79 + Exp11 blend grid

In [ ]:

main79_pred = main79_svcbag_val_prob >= 0.5
exp11_pred = exp11_val_prob_B >= 0.5
truth = y_val == 1

print('NEW MAIN79 accuracy:', main79_svcbag_val_acc)
print('NEW MAIN79 AUC:', main79_svcbag_val_auc)
print('EXP11 accuracy:', exp11_val['val_acc'])
print('EXP11 AUC:', exp11_val['val_auc'])

print('\nERROR OVERLAP — NEW MAIN79 vs EXP11')
print('Both correct:', int(np.sum((main79_pred == truth) & (exp11_pred == truth))))
print('New Main79 only:', int(np.sum((main79_pred == truth) & (exp11_pred != truth))))
print('Exp11 only:', int(np.sum((main79_pred != truth) & (exp11_pred == truth))))
print('Both wrong:', int(np.sum((main79_pred != truth) & (exp11_pred != truth))))

best_blend_bag = None
for alpha in np.linspace(0.0, 1.0, 101):
    p = alpha * main79_svcbag_val_prob + (1.0 - alpha) * exp11_val_prob_B
    acc = accuracy_score(y_val, p >= 0.5)
    auc = roc_auc_score(y_val, p)
    if best_blend_bag is None or acc > best_blend_bag['accuracy']:
        best_blend_bag = {'alpha': float(alpha), 'accuracy': float(acc), 'auc': float(auc)}

print('\n' + '=' * 72)
print('NEW MAIN79(SVC-BAG) + EXP11')
print('=' * 72)
print('Best transparent blend:', best_blend_bag)
print('Delta vs locked Main79+Exp11 accuracy:', best_blend_bag['accuracy'] - LOCKED_BLEND_ACC)
print('Delta vs locked Main79+Exp11 AUC:', best_blend_bag['auc'] - LOCKED_BLEND_AUC)

comparison = {
    'locked_main79': {
        'val_accuracy': LOCKED_MAIN79_ACC,
        'val_auc': LOCKED_MAIN79_AUC,
        'public_score': LOCKED_PUBLIC_MAIN79,
    },
    'locked_main79_exp11': {
        'val_accuracy': LOCKED_BLEND_ACC,
        'val_auc': LOCKED_BLEND_AUC,
        'public_score': LOCKED_PUBLIC_BLEND,
    },
    'bagged_svc_teacher': {
        'val_accuracy': float(teacher_val_acc_bag),
        'val_auc': float(teacher_val_auc_bag),
    },
    'main79_svcbag': {
        'val_accuracy': float(main79_svcbag_val_acc),
        'val_auc': float(main79_svcbag_val_auc),
    },
    'exp11': {
        'val_accuracy': float(exp11_val['val_acc']),
        'val_auc': float(exp11_val['val_auc']),
    },
    'main79_svcbag_exp11_blend': best_blend_bag,
}

with open(OUTPUT_DIR / 'validation_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(comparison, f, indent=2)


NEW MAIN79 accuracy: 0.9369070208728653
NEW MAIN79 AUC: 0.9802009246088195
EXP11 accuracy: 0.9060721062618596
EXP11 AUC: 0.9660166745693062

ERROR OVERLAP — NEW MAIN79 vs EXP11
Both correct: 1857
New Main79 only: 118
Exp11 only: 53
Both wrong: 80

NEW MAIN79(SVC-BAG) + EXP11
Best transparent blend: {'alpha': 0.86, 'accuracy': 0.9411764705882353, 'auc': 0.9821370712818082}
Delta vs locked Main79+Exp11 accuracy: 0.0023719165085388516
Delta vs locked Main79+Exp11 AUC: -0.0005294768452662746


In [ ]:
from pathlib import Path

OUT = Path("/content/main79_svcbag_fast_outputs")

print("OUTPUT FILES:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUT))

print("\nMAIN79 MODEL FILES:")
MODEL_DIR = OUT / "main79_validation_models"
if MODEL_DIR.exists():
    for p in sorted(MODEL_DIR.rglob("*")):
        if p.is_file():
            print(p.relative_to(MODEL_DIR))
else:
    print("MODEL DIRECTORY NOT FOUND")

OUTPUT FILES:
bagged_teacher_val_meta.npy
exp11_val_prob_B.npy
main66.py
main79_95_kaggle.py
main79_svcbag_val_labels.npy
main79_svcbag_val_predictions.npy
main79_svcbag_val_prob.npy
main79_svcbag_val_score.npy
main79_validation_models/main79_residual_seed123.pt
main79_validation_models/main79_residual_seed42.pt
main79_validation_models/main79_residual_seed777.pt
svc_bag_oof.npy
svc_bag_val.npy
validation_comparison.json

MAIN79 MODEL FILES:
main79_residual_seed123.pt
main79_residual_seed42.pt
main79_residual_seed777.pt


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/main79_svcbag_fast_outputs")
zip_base = Path("/content/MAIN79_SVCBAG_SECTION7_SAFE")

shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=src.parent,
    base_dir=src.name
)

print("SAFE ZIP CREATED:")
print(zip_base.with_suffix(".zip"))
print("Size MB:", round(zip_base.with_suffix(".zip").stat().st_size / 1024**2, 2))

SAFE ZIP CREATED:
/content/MAIN79_SVCBAG_SECTION7_SAFE.zip
Size MB: 13.57


## 8. FULL-DATA TEST INFERENCE — VERIFIED SVC-BAG + MAIN79 + EXP11

This section is the cleaned all-in-one version of the exact pipeline that produced the final submission.

- 3-member LinearSVC bootstrap bag: seeds 42, 123, 777; C=10; bootstrap fraction=1.0.
- Full-data Main66 teacher, followed by the exact Main79 residual architecture.
- Exp11 retrained on all 10,536 training samples using the validation-selected epoch count.
- Final blend uses the validation-selected transparent weight **alpha = 0.86**.

The SVC members and Main79 checkpoints are cached so that an interrupted run can resume without repeating completed full-data fits.


In [4]:
# ============================================================
# 3. RESUMABLE FULL-DATA SVC BAG
# ============================================================
def make_bootstrap_indices(n, seed, fraction=1.0):
    rng = np.random.RandomState(seed)
    m = max(1, int(round(fraction * n)))
    return rng.randint(0, n, size=m)

def fit_svc_bag_test_resumable(X_fit, y_fit, X_query, seeds=SVC_BAG_SEEDS):
    scores = []
    n = len(y_fit)
    member_dir = OUTPUT_DIR / 'full_test_svc_members'
    member_dir.mkdir(parents=True, exist_ok=True)

    for k, seed in enumerate(seeds, 1):
        cache = member_dir / f'svc_test_seed{seed}.npy'

        if cache.exists():
            score = np.load(cache).reshape(-1).astype(np.float32)
            if score.shape == (len(X_query),) and np.isfinite(score).all():
                print(f'[TEST] REUSING SVC {k}/{len(seeds)} seed={seed}')
                scores.append(score)
                continue

        boot = make_bootstrap_indices(n, seed, SVC_BOOTSTRAP_FRACTION)

        print(f'[TEST] TRAINING SVC {k}/{len(seeds)} seed={seed} ...')
        t0 = time.time()

        model = LinearSVC(
            C=SVC_C,
            class_weight='balanced',
            tol=1e-2,
            max_iter=500000,
            random_state=seed,
        )
        model.fit(X_fit[boot], y_fit[boot])
        score = model.decision_function(X_query).astype(np.float32)

        np.save(cache, score)
        scores.append(score)

        print(f'[TEST] SVC {k}/{len(seeds)} finished in {(time.time()-t0)/60:.2f} min')
        print('        range:', float(score.min()), float(score.max()))

        del model, boot
        gc.collect()

    return np.mean(np.vstack(scores), axis=0).astype(np.float32)

print('SVC bag function ready.')


SVC bag function ready.


In [5]:
# ============================================================
# 4. SECTION 8A — FULL-DATA MAIN66 REPRESENTATION + 3-SVC BAG
# ============================================================
print('=' * 72)
print('SECTION 8A — FULL-DATA MAIN66 + 3-SVC BAG')
print('=' * 72)

t0 = time.time()

(
    full_strings,
    full_transitions,
    full_tfidf,
    full_trans_vectorizer,
    full_compact_scaler,
    X_full_tfidf,
    X_full_compact,
    X_full_global,
) = main66.fit_global_representation(texts)

(
    test_strings_main66,
    test_transitions_main66,
    X_test_tfidf,
    X_test_compact,
    X_test_global,
) = main66.transform_global_representation(
    test_texts,
    full_tfidf,
    full_trans_vectorizer,
    full_compact_scaler,
)

print('Full global:', X_full_global.shape)
print('Test global:', X_test_global.shape)
print('Representation time:', (time.time()-t0)/60, 'min')

test_svc_bag = fit_svc_bag_test_resumable(
    X_full_global,
    labels,
    X_test_global,
)

np.save(OUTPUT_DIR / 'svc_bag_test_score.npy', test_svc_bag)

print('\nFULL-DATA SVC BAG COMPLETE')
print('Mean/std:', float(test_svc_bag.mean()), float(test_svc_bag.std()))


SECTION 8A — FULL-DATA MAIN66 + 3-SVC BAG
Full global: (10536, 591664)
Test global: (3000, 591664)
Representation time: 1.0303532958030701 min
[TEST] TRAINING SVC 1/3 seed=42 ...
[TEST] SVC 1/3 finished in 12.89 min
        range: -2.2361350059509277 91.86397552490234
[TEST] TRAINING SVC 2/3 seed=123 ...
[TEST] SVC 2/3 finished in 4.44 min
        range: -20.578035354614258 38.51399612426758
[TEST] TRAINING SVC 3/3 seed=777 ...
[TEST] SVC 3/3 finished in 7.35 min
        range: -50.775184631347656 32.82842254638672

FULL-DATA SVC BAG COMPLETE
Mean/std: 0.6701695322990417 1.6325887441635132


In [6]:
# ============================================================
# 5. SECTION 8B — OTHER MAIN66 COMPONENTS + TEST TEACHER
# ============================================================
print('=' * 72)
print('SECTION 8B — MAIN66 NON-SVC COMPONENTS + TEACHER')
print('=' * 72)

test_nbsvm = main66.train_nbsvm(
    full_strings, labels, test_strings_main66
)

test_hgb = main66.train_hgb(
    X_full_compact, labels, X_test_compact
)

test_local = main66.local_geometry_features(
    X_full_tfidf,
    labels,
    X_test_tfidf,
    k=main66.LOCAL_K,
)

test_meta_bag, test_raw_bag = make_teacher_meta(
    test_svc_bag,
    test_nbsvm,
    test_hgb,
    test_local,
    bag_oof_meta_scaler,
)

test_teacher_raw_bag = teacher_model_bag.decision_function(test_meta_bag).astype(np.float32)
test_teacher_bag = np.clip(
    test_teacher_raw_bag / teacher_scale_bag, -6.0, 6.0
).astype(np.float32)

test_meta_z_bag = np.clip(
    (test_meta_bag - meta_mean_bag) / meta_std_bag, -6.0, 6.0
).astype(np.float32)

# Exact all-row auxiliary meta:
# canonical training partition gets OOF-derived meta;
# canonical validation partition gets saved validation-derived meta.
meta_all_bag = np.empty((len(texts), 13), dtype=np.float32)
meta_all_bag[train_idx] = meta_train_z_bag
meta_all_bag[val_idx] = meta_val_z_bag

np.save(OUTPUT_DIR / 'bagged_test_teacher_score.npy', test_teacher_bag)
np.save(OUTPUT_DIR / 'test_meta_z_bag.npy', test_meta_z_bag)

print('Teacher test range:', float(test_teacher_bag.min()), float(test_teacher_bag.max()))
print('Test A/B by teacher threshold 0:', int((test_teacher_bag < 0).sum()), int((test_teacher_bag >= 0).sum()))
print('meta_all_bag:', meta_all_bag.shape)


SECTION 8B — MAIN66 NON-SVC COMPONENTS + TEACHER
Teacher test range: -0.9314612746238708 4.422540664672852
Test A/B by teacher threshold 0: 1050 1950
meta_all_bag: (10536, 13)


In [ ]:
# ============================================================
# SECTION 8C PREP — MAIN79 MODULE GLOBALS
# ============================================================
# The original Main79 script normally defines these globals inside main().
# We call train_one_seed() directly here, so define them explicitly.
main79.VOCAB_SIZE = max(int(t) for s in texts for t in s) + 1
main79.PAD_IDX = main79.VOCAB_SIZE
print("MAIN79 GLOBALS READY")
print("main79.VOCAB_SIZE =", main79.VOCAB_SIZE)
print("main79.PAD_IDX    =", main79.PAD_IDX)


In [11]:
# ============================================================
# 6. SECTION 8C — RESUMABLE FULL-DATA MAIN79 RESIDUAL ENSEMBLE
# ============================================================
print('=' * 72)
print('SECTION 8C — FULL-DATA MAIN79 RESIDUAL ENSEMBLE')
print('=' * 72)

FULL_MODEL_DIR = OUTPUT_DIR / 'main79_full_models'
FULL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
main79.OUT_DIR = FULL_MODEL_DIR

full_ds = main79.ResidualDataset(texts, labels, meta_all_bag)
full_loader = DataLoader(
    full_ds,
    batch_size=main79.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(main79.DEVICE.type == 'cuda'),
)

full_residuals = []

for seed in main79.MODEL_SEEDS:
    ckpt = FULL_MODEL_DIR / f'main79_residual_seed{seed}.pt'

    if ckpt.exists():
        print(f'\nREUSING existing Main79 seed {seed}: {ckpt}')
        model = main79.ResidualBiGRU(meta_all_bag.shape[1]).to(main79.DEVICE)
        state = torch.load(ckpt, map_location=main79.DEVICE)
        model.load_state_dict(state)
        model.eval()
    else:
        print(f'\nTRAINING full-data Main79 seed {seed} ...')
        model = main79.train_one_seed(
            seed,
            full_loader,
            meta_all_bag.shape[1]
        )

    r = main79.predict_residual(
        model,
        test_texts,
        test_meta_z_bag,
        test_teacher_bag
    )
    full_residuals.append(r)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

full_residual_mean_bag = np.mean(
    np.vstack(full_residuals), axis=0
).astype(np.float32)

main79_svcbag_test_score = (
    test_teacher_bag +
    main79.RESIDUAL_WEIGHT * full_residual_mean_bag
).astype(np.float32)

main79_svcbag_test_prob = (
    1.0 / (1.0 + np.exp(-main79_svcbag_test_score))
).astype(np.float32)

assert main79_svcbag_test_prob.shape == (3000,)
assert np.isfinite(main79_svcbag_test_prob).all()

np.save(OUTPUT_DIR / 'main79_svcbag_test_score.npy', main79_svcbag_test_score)
np.save(OUTPUT_DIR / 'main79_svcbag_test_prob.npy', main79_svcbag_test_prob)

print('\nNEW MAIN79 SVC-BAG TEST PREDICTION READY')
print('A:', int((main79_svcbag_test_prob < 0.5).sum()))
print('B:', int((main79_svcbag_test_prob >= 0.5).sum()))


SECTION 8C — FULL-DATA MAIN79 RESIDUAL ENSEMBLE

TRAINING full-data Main79 seed 42 ...

TRAINING MAIN79 RESIDUAL — SEED 42
Parameters: 1264258
epoch 01/6 | train_acc=0.87130 | loss=0.46863 | time=0.09 min
epoch 02/6 | train_acc=0.92056 | loss=0.26997 | time=0.06 min
epoch 03/6 | train_acc=0.92322 | loss=0.21907 | time=0.08 min
epoch 04/6 | train_acc=0.92654 | loss=0.20814 | time=0.06 min
epoch 05/6 | train_acc=0.93128 | loss=0.19577 | time=0.06 min
epoch 06/6 | train_acc=0.93603 | loss=0.18393 | time=0.07 min
Saved: /content/main79_svcbag_fast_outputs/main79_full_models/main79_residual_seed42.pt

TRAINING full-data Main79 seed 123 ...

TRAINING MAIN79 RESIDUAL — SEED 123
Parameters: 1264258
epoch 01/6 | train_acc=0.88753 | loss=0.45319 | time=0.06 min
epoch 02/6 | train_acc=0.92084 | loss=0.26279 | time=0.06 min
epoch 03/6 | train_acc=0.92379 | loss=0.22398 | time=0.07 min
epoch 04/6 | train_acc=0.92445 | loss=0.21347 | time=0.07 min
epoch 05/6 | train_acc=0.92616 | loss=0.20522 | time

In [14]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# ============================================================
# FULL-DATA EXP11 RETRAIN + TEST PREDICTION
# Same Exp11 architecture. Use the validation-selected epoch count.
# ============================================================
best_epoch = 5  # verified from the completed Section-7 notebook

all_docs = [[str(t) for t in s] for s in texts]
test_docs = [[str(t) for t in s] for s in test_texts]

full_vocab = build_vocab(all_docs)
tok_all = encode_docs(all_docs, full_vocab)
tok_test = encode_docs(test_docs, full_vocab)

all_strings_exp11 = [' '.join(d) for d in all_docs]
test_strings_exp11 = [' '.join(d) for d in test_docs]

vec_full = TfidfVectorizer(
    ngram_range=(1,6),
    min_df=EXP11_TFIDF_MIN_DF,
    max_features=EXP11_TFIDF_MAX_FEATURES,
    sublinear_tf=True,
    token_pattern=r'(?u)\S+',
    dtype=np.float32,
)

A = vec_full.fit_transform(all_strings_exp11)
T = vec_full.transform(test_strings_exp11)
ncomp = min(EXP11_SVD_DIM, A.shape[1] - 1)
svd_full = TruncatedSVD(n_components=ncomp, random_state=EXP11_SEED)
A_svd = svd_full.fit_transform(A).astype(np.float32)
T_svd = svd_full.transform(T).astype(np.float32)

struct_all_raw = make_exp11_structural(all_docs)
struct_test_raw = make_exp11_structural(test_docs)
struct_scaler = StandardScaler()
struct_all = struct_scaler.fit_transform(struct_all_raw).astype(np.float32)
struct_test = struct_scaler.transform(struct_test_raw).astype(np.float32)

branches_all = {'tfidf': A_svd, 'structural': struct_all}
branches_test = {'tfidf': T_svd, 'structural': struct_test}

class TestDataset(Dataset):
    def __init__(self, tok, b):
        self.tok = tok
        self.b = b
    def __len__(self):
        return len(self.tok)
    def __getitem__(self, i):
        return (
            torch.tensor(self.tok[i], dtype=torch.long),
            {k: torch.tensor(v[i], dtype=torch.float32) for k, v in self.b.items()},
        )

class FullExpDataset(Dataset):
    def __init__(self, tok, y, b):
        self.tok = tok
        self.y = np.asarray(y, dtype=np.float32)
        self.b = b
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return (
            torch.tensor(self.tok[i], dtype=torch.long),
            torch.tensor(self.y[i], dtype=torch.float32),
            {k: torch.tensor(v[i], dtype=torch.float32) for k, v in self.b.items()},
        )

full_y_A = (labels == 0).astype(np.float32)
full_ds_exp11 = FullExpDataset(tok_all, full_y_A, branches_all)
full_loader_exp11 = DataLoader(full_ds_exp11, batch_size=EXP11_BATCH_SIZE, shuffle=True)
test_ds_exp11 = TestDataset(tok_test, branches_test)
test_loader_exp11 = DataLoader(test_ds_exp11, batch_size=EXP11_BATCH_SIZE, shuffle=False)

dims_full = {k: v.shape[1] for k, v in branches_all.items()}
device_exp11 = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
full_model_exp11 = FusionModel(len(full_vocab), dims_full).to(device_exp11)

pos_weight = float((full_y_A == 0).sum() / max(1, (full_y_A == 1).sum()))
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device_exp11))
optimizer = torch.optim.AdamW(full_model_exp11.parameters(), lr=EXP11_LR, weight_decay=EXP11_WEIGHT_DECAY)

print('Exp11 full-data training epochs:', best_epoch)
for epoch in range(1, best_epoch + 1):
    full_model_exp11.train()
    losses = []
    for tok, y, b in full_loader_exp11:
        tok = tok.to(device_exp11)
        y = y.to(device_exp11)
        b = {k: v.to(device_exp11) for k, v in b.items()}
        optimizer.zero_grad(set_to_none=True)
        logits, _ = full_model_exp11(tok, b)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(full_model_exp11.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.item()))
    print(f'Exp11 full epoch {epoch:02d}/{best_epoch} loss={np.mean(losses):.6f}')

full_model_exp11.eval()
test_prob_A = []
with torch.no_grad():
    for tok, b in test_loader_exp11:
        tok = tok.to(device_exp11)
        b = {k: v.to(device_exp11) for k, v in b.items()}
        logits, _ = full_model_exp11(tok, b)
        test_prob_A.extend(torch.sigmoid(logits).cpu().numpy())

exp11_test_prob_A = np.asarray(test_prob_A, dtype=np.float32)
exp11_test_prob_B = 1.0 - exp11_test_prob_A
assert exp11_test_prob_B.shape == (3000,)
assert np.isfinite(exp11_test_prob_B).all()

np.save(OUTPUT_DIR / 'exp11_test_prob_B.npy', exp11_test_prob_B)
print('Exp11 test A/B:', int((exp11_test_prob_B < 0.5).sum()), int((exp11_test_prob_B >= 0.5).sum()))

Exp11 full-data training epochs: 5
Exp11 full epoch 01/5 loss=0.460976
Exp11 full epoch 02/5 loss=0.314266
Exp11 full epoch 03/5 loss=0.255991
Exp11 full epoch 04/5 loss=0.195876
Exp11 full epoch 05/5 loss=0.141869
Exp11 test A/B: 1157 1843


In [ ]:
# ============================================================
# REPRODUCIBILITY CHECK — VALIDATION REFERENCE
# ============================================================
REFERENCE_MAIN79_ACC = 0.9369070208728653
REFERENCE_BLEND_ACC = 0.9411764705882353
REFERENCE_BLEND_AUC = 0.9821370712818082
REFERENCE_ALPHA = 0.86

print("Observed Main79 SVC-bag validation accuracy:", main79_svcbag_val_acc)
print("Reference:", REFERENCE_MAIN79_ACC)
print("Observed best blend:", best_blend_bag)
print("Reference alpha:", REFERENCE_ALPHA)

if abs(main79_svcbag_val_acc - REFERENCE_MAIN79_ACC) > 1e-6:
    print("WARNING: Main79 validation accuracy differs from the verified run.")
if abs(best_blend_bag['accuracy'] - REFERENCE_BLEND_ACC) > 1e-6:
    print("WARNING: blend validation accuracy differs from the verified run.")
if abs(best_blend_bag['auc'] - REFERENCE_BLEND_AUC) > 1e-6:
    print("WARNING: blend AUC differs from the verified run.")
if abs(best_blend_bag['alpha'] - REFERENCE_ALPHA) > 1e-12:
    raise RuntimeError(f"Unexpected blend alpha: {best_blend_bag['alpha']} (expected {REFERENCE_ALPHA})")


In [ ]:

def write_submission(filename, prob):
    prob = np.asarray(prob).reshape(-1)
    assert len(prob) == 3000
    assert np.isfinite(prob).all()
    pred = np.where(prob >= 0.5, 'B', 'A')
    sub = pd.DataFrame({'id': test_ids, 'label': pred})
    assert len(sub) == 3000
    assert sub['id'].nunique() == 3000
    assert list(sub.columns) == ['id', 'label']
    assert set(sub['label'].unique()).issubset({'A', 'B'})
    path = OUTPUT_DIR / filename
    sub.to_csv(path, index=False)
    check = pd.read_csv(path)
    assert len(check) == 3000
    assert check['id'].is_unique
    assert set(check['label'].unique()).issubset({'A', 'B'})
    print(f'CREATED {filename} | A={int((pred=="A").sum())} | B={int((pred=="B").sum())}')
    return path

alpha = best_blend_bag['alpha']
blend_test_prob = alpha * main79_svcbag_test_prob + (1.0 - alpha) * exp11_test_prob_B

sub_main79 = write_submission('submission_main79_svcbag.csv', main79_svcbag_test_prob)
sub_exp11 = write_submission('submission_exp11_reproduced.csv', exp11_test_prob_B)
sub_blend = write_submission('submission_main79_svcbag_exp11_blend.csv', blend_test_prob)

final_report = {
    'experiment': 'Main79 with bootstrap-bagged LinearSVC replacing the single Main66 SVC',
    'svc_bag_size': SVC_BAG_SIZE,
    'svc_bag_seeds': SVC_BAG_SEEDS,
    'svc_bootstrap_fraction': SVC_BOOTSTRAP_FRACTION,
    'svc_oof_folds': SVC_OOF_FOLDS,
    'main79_validation_accuracy': float(main79_svcbag_val_acc),
    'main79_validation_auc': float(main79_svcbag_val_auc),
    'main79_delta_vs_locked_accuracy': float(main79_svcbag_val_acc - LOCKED_MAIN79_ACC),
    'main79_delta_vs_locked_auc': float(main79_svcbag_val_auc - LOCKED_MAIN79_AUC),
    'exp11_validation_accuracy': float(exp11_val['val_acc']),
    'exp11_validation_auc': float(exp11_val['val_auc']),
    'exp11_best_epoch': int(exp11_val['best_epoch']),
    'blend_alpha': float(alpha),
    'blend_validation_accuracy': float(best_blend_bag['accuracy']),
    'blend_validation_auc': float(best_blend_bag['auc']),
    'blend_delta_vs_locked_accuracy': float(best_blend_bag['accuracy'] - LOCKED_BLEND_ACC),
    'blend_delta_vs_locked_auc': float(best_blend_bag['auc'] - LOCKED_BLEND_AUC),
    'test_counts': {
        'main79_svcbag_A': int((main79_svcbag_test_prob < 0.5).sum()),
        'main79_svcbag_B': int((main79_svcbag_test_prob >= 0.5).sum()),
        'exp11_A': int((exp11_test_prob_B < 0.5).sum()),
        'exp11_B': int((exp11_test_prob_B >= 0.5).sum()),
        'blend_A': int((blend_test_prob < 0.5).sum()),
        'blend_B': int((blend_test_prob >= 0.5).sum()),
    },
    'locked_references': {
        'main79_val_accuracy': LOCKED_MAIN79_ACC,
        'main79_val_auc': LOCKED_MAIN79_AUC,
        'main79_public': LOCKED_PUBLIC_MAIN79,
        'main79_exp11_val_accuracy': LOCKED_BLEND_ACC,
        'main79_exp11_val_auc': LOCKED_BLEND_AUC,
        'main79_exp11_public': LOCKED_PUBLIC_BLEND,
    },
    'submission_files': [
        sub_main79.name,
        sub_exp11.name,
        sub_blend.name,
    ],
}

with open(OUTPUT_DIR / 'final_experiment_report.json', 'w', encoding='utf-8') as f:
    json.dump(final_report, f, indent=2)

summary_lines = [
    'MAIN79 SVC-BAG FINAL RUN',
    '=' * 72,
    f'New Main79 validation accuracy: {main79_svcbag_val_acc:.6f}',
    f'New Main79 validation AUC:      {main79_svcbag_val_auc:.6f}',
    f'Locked Main79 validation:       {LOCKED_MAIN79_ACC:.6f}',
    f'New Main79 delta:                {main79_svcbag_val_acc - LOCKED_MAIN79_ACC:+.6f}',
    '',
    f'New Main79+Exp11 blend alpha:   {alpha:.2f}',
    f'New blend validation accuracy:   {best_blend_bag["accuracy"]:.6f}',
    f'New blend validation AUC:        {best_blend_bag["auc"]:.6f}',
    f'Locked old blend validation:     {LOCKED_BLEND_ACC:.6f}',
    f'New blend delta:                 {best_blend_bag["accuracy"] - LOCKED_BLEND_ACC:+.6f}',
    '',
    'TEST COUNTS',
    f'Main79 SVC-bag: A={int((main79_svcbag_test_prob < 0.5).sum())}, B={int((main79_svcbag_test_prob >= 0.5).sum())}',
    f'Exp11:          A={int((exp11_test_prob_B < 0.5).sum())}, B={int((exp11_test_prob_B >= 0.5).sum())}',
    f'Blend:          A={int((blend_test_prob < 0.5).sum())}, B={int((blend_test_prob >= 0.5).sum())}',
    '',
    'SUBMISSIONS:',
    f'  {sub_main79.name}',
    f'  {sub_exp11.name}',
    f'  {sub_blend.name}',
]

(OUTPUT_DIR / 'RUN_SUMMARY.txt').write_text('\n'.join(summary_lines), encoding='utf-8')
print('\n'.join(summary_lines))


## 10. Package outputs
After Run All, send back `/content/main79_svcbag_fast_outputs.zip`.


In [ ]:

# ============================================================
# 10. PACKAGE OUTPUTS
# ============================================================
OUTPUT_ZIP = CONTENT / 'main79_svcbag_outputs.zip'
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

with zipfile.ZipFile(OUTPUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob('*'):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUTPUT_DIR))

print('\n' + '=' * 72)
print('FINAL PACKAGE CREATED')
print('=' * 72)
print('Output directory:', OUTPUT_DIR)
print('ZIP:', OUTPUT_ZIP)
print('ZIP size (MB):', OUTPUT_ZIP.stat().st_size / (1024**2))
print('\nFiles in output:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(OUTPUT_DIR))
